In [1]:
import random
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import xgboost as xgb
import mlflow
import optuna
import datetime

from sklearn.model_selection import (
    train_test_split,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import roc_auc_score, f1_score

/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("Bank-Customer-Churn-Prediction-Experiment")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1757553436595, experiment_id='1', last_update_time=1757553436595, lifecycle_stage='active', name='Bank-Customer-Churn-Prediction-Experiment', tags={}>

# Data preprocessing

In [3]:
data_path = "../data/Customer-Churn-Records.csv"


def clean_data(data_path):
    df = pd.read_csv(data_path)
    df.drop(columns=["RowNumber", "CustomerId", "Surname", "Complain"], inplace=True)
    df.head()

    return df


def split_data(df):
    # Train/val/test stratified split of ratio 0.8/0.1/0.1
    labels = df.Exited.values
    del df["Exited"]

    X_train, X_vtest, y_train, y_vtest = train_test_split(
        df, labels, test_size=0.2, random_state=seed, stratify=labels
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_vtest, y_vtest, test_size=0.5, random_state=seed, stratify=y_vtest
    )

    return X_train, y_train, X_val, y_val, X_test, y_test

In [4]:
cat = [
    "Geography",
    "Gender",
    "NumOfProducts",
    "HasCrCard",
    "IsActiveMember",
    "Satisfaction Score",
    "Card Type",
]

num = ["CreditScore", "Age", "Tenure", "Balance", "EstimatedSalary", "Point Earned"]


def preprocess_data(X_train, X_val, X_test):
    preprocessor = ColumnTransformer(
        [
            ("oh", OneHotEncoder(handle_unknown="ignore"), cat),
            ("scaler", StandardScaler(), num),
        ]
    )

    X_train = preprocessor.fit_transform(X_train)
    X_val = preprocessor.transform(X_val)
    X_test = preprocessor.transform(X_test)

    return X_train, X_val, X_test, preprocessor

In [5]:
records = clean_data(data_path)
X_train, y_train, X_val, y_val, X_test, y_test = split_data(records)
X_train, X_val, X_test, pp = preprocess_data(X_train, X_val, X_test)

# Model Evaluation and Hyperparameters Tuning

In [6]:
sampler = optuna.samplers.TPESampler(seed=seed)

In [ ]:
def xgb_objective(trial):
    with mlflow.start_run(nested=True):
        train = xgb.DMatrix(X_train, label=y_train)
        valid = xgb.DMatrix(X_val, label=y_val)

        params = {
            "n_estimators": trial.suggest_int("n_estimators", 100, 5000),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 1.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-9, 100.0, log=True),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-9, 100.0, log=True),
            "subsample": trial.suggest_float("subsample", 0.1, 1.0),
            "max_depth": trial.suggest_int("max_depth", 1, 12),
            "max_delta_step": trial.suggest_int("max_delta_step", 0, 10),
            "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
            "gamma": trial.suggest_float("gamma", 1e-9, 0.5, log=True),
            "scale_pos_weight": trial.suggest_float(
                "scale_pos_weight", 1e-6, 500.0, log=True
            ),
            "seed": seed,
        }

        model = xgb.train(
            params,
            train,
            evals=[(valid, "validation")],
            early_stopping_rounds=300,
            verbose_eval=False,
        )

        preds = model.predict(valid)
        pred_labels = np.clip(np.rint(preds), 0, 1)

        f1 = f1_score(y_val, pred_labels)
        roc_auc = roc_auc_score(y_val, pred_labels)

        mlflow.log_metric("roc_auc", roc_auc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_params(params)

    return roc_auc


def train_best_xgb_model(X_train, y_train, best_params):
    train = xgb.DMatrix(X_train, label=y_train)

    model = xgb.train(best_params, train)

    return model


def plot_feature_importance(model, feat_names=None):
    """
    Plots feature importance for an XGBoost model.

    Args:
    - model: A trained XGBoost model

    Returns:
    - fig: The matplotlib figure object
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    importance_type = "gain"
    if feat_names is not None:
        model.feature_names = list(feat_names)

    xgb.plot_importance(
        model,
        importance_type=importance_type,
        ax=ax,
        title=f"Feature Importance based on {importance_type}",
    )
    plt.tight_layout()
    plt.close(fig)

    return fig


def hyperparameter_tuning(X_train, y_train, feat_names=None):
    with mlflow.start_run(
        run_name=f"xgboost_hyperparameter_tuning_{datetime.datetime.now().date()}",
        nested=True,
    ):
        mlflow.set_tag("model", "xgboost")
        study_xgb = optuna.create_study(direction="maximize", sampler=sampler)
        study_xgb.optimize(xgb_objective, n_trials=200)

        print("Number of finished trials:", len(study_xgb.trials))
        print("Best value:", study_xgb.best_value)

        mlflow.log_params(study_xgb.best_params)

        xgb_model = train_best_xgb_model(X_train, y_train, study_xgb.best_params)

        mlflow.xgboost.log_model(
            xgb_model=xgb_model,
            name="mlflow_model",
            input_example=X_train[:5],
            registered_model_name="XGBoostChurnModel",
        )

        importances = plot_feature_importance(
            xgb_model,
            feat_names=feat_names,
        )
        mlflow.log_figure(figure=importances, artifact_file="feature_importances.png")

In [ ]:
hyperparameter_tuning(
    X_train=X_train, y_train=y_train, feat_names=pp.get_feature_names_out()
)

[I 2025-09-11 13:56:46,269] A new study created in memory with name: no-name-ec75fcd1-8baf-4e4a-af3a-5b075fe59257
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [13:56:46] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-11 13:56:46,338] Trial 0 finished with value: 0.5 and parameters: {'n_estimators': 605, 'learning_rate': 0.6384190143734894, 'reg_lambda': 0.00036122358368275917, 'reg_alpha': 1.233196056245317, 'subsample': 0.3880446409275506, 'max_depth': 11, 'max_delta_step': 4, 'min_child_weight': 1, 'gamma': 0.07514336776922699, 'scale_pos_weight': 6.224436821233275e-06}. Best is trial 0 with value: 0.5.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12

🏃 View run rebellious-hawk-309 at: http://localhost:5000/#/experiments/1/runs/519dc1241ab34f33b6aa557b7010c58d
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run upbeat-pug-400 at: http://localhost:5000/#/experiments/1/runs/11893a5578f64d9694f417c903644e68
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run capable-koi-954 at: http://localhost:5000/#/experiments/1/runs/b5a7291aba094fab9e98938e85815228
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run flawless-worm-431 at: http://localhost:5000/#/experiments/1/runs/1caa95e634374c20bfc33b37762460a3
🧪 View experiment at: http://localhost:5000/#/experiments/1


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [13:56:46] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-11 13:56:46,598] Trial 4 finished with value: 0.5 and parameters: {'n_estimators': 515, 'learning_rate': 0.9403275426747112, 'reg_lambda': 1.3091178763820828e-05, 'reg_alpha': 1.1941630189212588e-05, 'subsample': 0.8315196105317524, 'max_depth': 12, 'max_delta_step': 10, 'min_child_weight': 8, 'gamma': 1.8753004091551739e-06, 'scale_pos_weight': 5.325620674356389e-06}. Best is trial 1 with value: 0.7578332840673958.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [13:56:46] WARNING: /Users/runner/min

🏃 View run enchanting-calf-445 at: http://localhost:5000/#/experiments/1/runs/621aa6f55dee48c5a01db745a5ff02cf
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run salty-dove-420 at: http://localhost:5000/#/experiments/1/runs/ac37ce3170e046a2a88ee854061795d3
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run loud-worm-158 at: http://localhost:5000/#/experiments/1/runs/4ae1f79ec60a4230b405c6995fec1405
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run spiffy-mule-759 at: http://localhost:5000/#/experiments/1/runs/c395fcc0a50c4d3cac98c02ee40eb52f
🧪 View experiment at: http://localhost:5000/#/experiments/1


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [13:56:46] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-11 13:56:46,863] Trial 8 finished with value: 0.7236427234210268 and parameters: {'n_estimators': 3184, 'learning_rate': 0.24631869668246867, 'reg_lambda': 9.998888199778013e-05, 'reg_alpha': 0.008000900710226721, 'subsample': 0.6258828807307902, 'max_depth': 11, 'max_delta_step': 0, 'min_child_weight': 3, 'gamma': 0.18518294724864245, 'scale_pos_weight': 55.51085953194902}. Best is trial 1 with value: 0.7578332840673958.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [13:56:46] WARNING: /Users/runn

...

/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [13:57:02] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  self.starting_round = model.num_boosted_rounds()
[I 2025-09-11 13:57:02,955] Trial 197 finished with value: 0.7745590698590995 and parameters: {'n_estimators': 1767, 'learning_rate': 0.1266453298795145, 'reg_lambda': 0.17402816869102053, 'reg_alpha': 1.3524196580795899e-09, 'subsample': 0.8860803768727019, 'max_depth': 8, 'max_delta_step': 8, 'min_child_weight': 8, 'gamma': 9.85696343092344e-09, 'scale_pos_weight': 4.6375980369851595}. Best is trial 141 with value: 0.8018400827667751.
/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/callback.py:386: UserWarning: [13:57:02] WARNING: /Users/

🏃 View run thoughtful-shark-30 at: http://localhost:5000/#/experiments/1/runs/6c7a5a0e84fb433b8b06d3c63f105a0f
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run skillful-pig-204 at: http://localhost:5000/#/experiments/1/runs/28afb124f1c245c4bccda9945e1edf7a
🧪 View experiment at: http://localhost:5000/#/experiments/1
🏃 View run sassy-shad-475 at: http://localhost:5000/#/experiments/1/runs/e7222f0b88ae4827ba5d08155d7f4b29
🧪 View experiment at: http://localhost:5000/#/experiments/1
Number of finished trials: 200
Best value: 0.8018400827667751


/Users/hmnguyen1067/Downloads/Github/Bank-Customer-Churn-Prediction-Pipeline/.pixi/envs/default/lib/python3.12/site-packages/xgboost/training.py:183: UserWarning: [13:57:03] WARNING: /Users/runner/miniforge3/conda-bld/xgboost-split_1755048541311/work/src/learner.cc:738: 
Parameters: { "n_estimators" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2025/09/11 13:57:03 WARNING mlflow.models.signature: Failed to infer the model signature from the input example. Reason: ValueError('data did not contain feature names, but the following fields are expected: oh__Geography_France, oh__Geography_Germany, oh__Geography_Spain, oh__Gender_Female, oh__Gender_Male, oh__NumOfProducts_1, oh__NumOfProducts_2, oh__NumOfProducts_3, oh__NumOfProducts_4, oh__HasCrCard_0, oh__HasCrCard_1, oh__IsActiveMember_0, oh__IsActiveMember_1, oh__Satisfaction Score_1, oh__Satisfaction Score_2, oh__Satisfaction Score_3, oh__Satisfaction Score_4, oh__Satisfaction Score_5, oh__Card Type_DIAMOND, oh__Card Type

🏃 View run xgboost_hyperparameter_tuning_2025-09-11 at: http://localhost:5000/#/experiments/1/runs/3f0a090307154cb3940a683447b443e5
🧪 View experiment at: http://localhost:5000/#/experiments/1


Created version '4' of model 'XGBoostChurnModel'.


In [9]:
# run_name_to_find = "xgboost_hyperparameter_tuning"
# artifact_path = "mlflow_model"

# runs = mlflow.search_runs(filter_string=f'tags."mlflow.runName" = "{run_name_to_find}"')
# model_uri = f"runs:/{runs.run_id[0]}/{artifact_path}"

# loaded_model = mlflow.xgboost.load_model(model_uri=model_uri)
# preds = loaded_model.predict(xgb.DMatrix(X_test))
# pred_labels = np.clip(np.rint(preds), 0, 1)

In [14]:
mlflow.get_artifact_uri("mlflow_model")

2025/09/11 13:08:43 WARNING mlflow.tracking.fluent: No active run found. A new active run will be created. If this is not intended, please create a run using `mlflow.start_run()` first.


'mlflow-artifacts:/1/74348e770bd94e338e3ec46c4656e9be/artifacts/mlflow_model'

In [ ]:
loaded_model = mlflow.xgboost.load_model(f"models:/XGBoostChurnModel/latest")